<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img
 src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/alx-courses/aice/assets/Content_page_banner_blue_dots.png"
 alt="ALX Content Header"
 class="full-width-image"
/>
</div>

# Subqueries and CTEs

In this exercise, we will apply subqueries and CTEs in different parts of a query and for different use cases. Ensure that you have downloaded the database file, Northwind.db.

## Learning objectives

By the end of this train, you should:
- Know how to use CTEs to simplify subqueries.
- Understand when to use subqueries and when to use CTEs by comparing their performance and readability.

First, let's load our sample database:

In [1]:
# Load and activate the SQL extension to allow us to execute SQL in a Jupyter notebook.
%load_ext sql


In [2]:
# Load the Northwind database stored in your local machine. 
# Make sure the file is saved in the same folder as this notebook.
%sql sqlite:///Northwind.db
    

'Connected: @Northwind.db'

In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

Here is a view of all of our tables in the database:

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Northwind_ERD.png"  style="width:500px";/>
<br>
<br>
    <em>Figure 1: Northwind ERD</em>
</div>

## Exercise

Run the necessary queries that will provide us with the following information. Compare your queries with the solutions at the end of this notebook.

### Exercise 1

Retrieve product details from products that have been ordered by customers from the UK.

In [23]:
%%sql


-- SUBQUERY SOLUTION

SELECT
    o.*
FROM
    orders o
WHERE 
    o.CustomerID IN(
        SELECT
            c.CustomerID
        FROM
            customers c
        WHERE 
            c.Country = "UK"
        GROUP BY 
            c.CustomerID
        ORDER BY COUNT(*) DESC
    )
LIMIT 10;


-- CTE SOLUTION

WITH uk_customers AS (
    SELECT
        c.CustomerID
    FROM
        customers c
    WHERE 
        c.Country = "UK"
    GROUP BY 
        c.CustomerID
    ORDER BY COUNT(*) DESC
)

SELECT
    o.*
FROM
    orders o
JOIN
    uk_customers u
ON
    o.CustomerID = u.CustomerID
LIMIT 10;

 * sqlite:///Northwind.db
Done.
Done.


OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
10289,BSBEV,7,2016-08-26,2016-09-23,2016-08-28,3,19.75,B-s Beverages,Fauntleroy Circus,London,British Isles,EC2 5NT,UK
10315,ISLAT,4,2016-09-26,2016-10-24,2016-10-03,2,21,Island Trading,Garden House Crowther Way,Cowes,British Isles,PO31 7PJ,UK
10318,ISLAT,8,2016-10-01,2016-10-29,2016-10-04,2,16.5,Island Trading,Garden House Crowther Way,Cowes,British Isles,PO31 7PJ,UK
10321,ISLAT,3,2016-10-03,2016-10-31,2016-10-11,2,12.5,Island Trading,Garden House Crowther Way,Cowes,British Isles,PO31 7PJ,UK
10355,AROUT,6,2016-11-15,2016-12-13,2016-11-20,1,22.5,Around the Horn,Brook Farm Stratford St. Mary,Colchester,British Isles,CO7 6JX,UK
10359,SEVES,5,2016-11-21,2016-12-19,2016-11-26,3,61.5,Seven Seas Imports,90 Wadhurst Rd.,London,British Isles,OX15 4NB,UK
10364,EASTC,1,2016-11-26,2017-01-07,2016-12-04,1,18.75,Eastern Connection,35 King George,London,British Isles,WX3 6FW,UK
10377,SEVES,1,2016-12-09,2017-01-06,2016-12-13,3,20,Seven Seas Imports,90 Wadhurst Rd.,London,British Isles,OX15 4NB,UK
10383,AROUT,8,2016-12-16,2017-01-13,2016-12-18,3,23.75,Around the Horn,Brook Farm Stratford St. Mary,Colchester,British Isles,CO7 6JX,UK
10388,SEVES,2,2016-12-19,2017-01-16,2016-12-20,1,28.75,Seven Seas Imports,90 Wadhurst Rd.,London,British Isles,OX15 4NB,UK


### Exercise 2


Find out the names of customers who have ordered products of more than the average order value.

In [24]:
%%sql

WITH Avg_orders AS (
    SELECT
        AVG(od.UnitPrice * od.Quantity) AS avg_value
    FROM "Order Details" AS od
)

SELECT DISTINCT
    c.CompanyName
FROM
    customers AS c
JOIN
    orders AS o
ON
    c.CustomerID = o.CustomerID
JOIN
    "Order Details" AS od
ON
    o.OrderID = od.OrderID
WHERE
    (od.UnitPrice * od.Quantity) > (SELECT avg_value FROM Avg_orders);

 * sqlite:///Northwind.db
Done.


CompanyName
Toms Spezialitäten
Hanari Carnes
Suprêmes délices
Richter Supermarkt
HILARION-Abastos
Ernst Handel
Ottilies Käseladen
Blondesddsl père et fils
Frankenversand
GROSELLA-Restaurante


### Exercise 3


Write a CTE to find the most ordered product by each customer.

In [36]:
%%sql

WITH most_ordered_products AS (
    SELECT customers.CustomerID, OrderDetails.ProductID, COUNT(*) AS order_count
    FROM customers
    JOIN orders ON customers.CustomerID = orders.CustomerID
    JOIN "Order Details" AS OrderDetails ON orders.OrderID = OrderDetails.OrderID
    GROUP BY customers.CustomerID, OrderDetails.ProductID
)
SELECT customers.CompanyName, products.ProductName, max_order_count
FROM (
    SELECT CustomerID, MAX(order_count) AS max_order_count
    FROM most_ordered_products
    GROUP BY CustomerID
) AS max_order_count
JOIN most_ordered_products ON max_order_count.CustomerID = most_ordered_products.CustomerID AND max_order_count.max_order_count = most_ordered_products.order_count
JOIN customers ON most_ordered_products.CustomerID = customers.CustomerID
JOIN products ON most_ordered_products.ProductID = products.ProductID;

 * sqlite:///Northwind.db
Done.


CompanyName,ProductName,max_order_count
Alfreds Futterkiste,Grandma's Boysenberry Spread,79
Alfreds Futterkiste,Wimmers gute Semmelknödel,79
Ana Trujillo Emparedados y helados,Ipoh Coffee,111
Antonio Moreno Taquería,Ravioli Angelo,90
Around the Horn,Perth Pasties,89
Around the Horn,Escargots de Bourgogne,89
Berglunds snabbköp,Camembert Pierrot,99
Blauer See Delikatessen,Louisiana Hot Spiced Okra,94
Blondesddsl père et fils,Rhönbräu Klosterbier,101
Bólido Comidas preparadas,Chef Anton's Cajun Seasoning,106


### Exercise 4

Using a CTE, list employees who have more than the average number of reports.

In [37]:
%%sql

WITH avg_reports AS (
    SELECT AVG(report_count) AS average_count
    FROM (
        SELECT COUNT(*) AS report_count
        FROM employees
        JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
        GROUP BY employees.EmployeeID
    ) AS report_counts
)
SELECT employees.*
FROM employees
JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
GROUP BY employees.EmployeeID
HAVING COUNT(*) > (SELECT average_count FROM avg_reports);

 * sqlite:///Northwind.db
Done.


[(2, 'Fuller', 'Andrew', 'Vice President, Sales', 'Dr.', '1972-02-19', '2012-08-14', '908 W. Capital Way', 'Tacoma', 'North America', '98401', 'USA', '(206) 555-9482', '3457', b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x04\x00\x00\x00P\x00\x00\xff\xee\x00&Adobe\x00d\xc0\ ... (35229 characters truncated) ... \xee\xbd\xe6\xbf>w\xed4o\xed\xc4\xd2\x1b\x9e\xb4=\xfd\xa7\xdb\xfcO\xf6O\xec\xfdi\xeah\xf5\x9a=8\xfe\xcd\xdf\xd7\xfb\xe2l\xdf\xf2k\xfe\xa7\xff\xd9\x00', 'Andrew received his BTS commercial in 1974 and a Ph.D. in international marketing from the University of Dallas in 1981.  He is fluent in French and  ... (150 characters truncated) ... es in March 1993.  Andrew is a member of the Sales Management Roundtable, the Seattle Chamber of Commerce, and the Pacific Rim Importers Association.', None, 'http://accweb/emmployees/fuller.bmp')]

## Solutions

### Exercise 1

SQL solution with a subquery:

In [ ]:
%%sql

SELECT customers.*
FROM customers
WHERE customers.CustomerID IN (
    SELECT orders.CustomerID
    FROM orders
    GROUP BY orders.CustomerID
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

SQL solution with a CTE:

In [ ]:
%%sql

WITH most_orders AS (
    SELECT orders.CustomerID
    FROM orders
    GROUP BY orders.CustomerID
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
SELECT customers.*
FROM customers
JOIN most_orders
ON customers.CustomerID = most_orders.CustomerID;

Both solutions will return the customer with the most orders. The subquery solution nests the logic inside the main query, which can become difficult to read for more complex queries. The CTE solution separates the logic into a different part of the query, which can be more readable, especially for more complex queries.

### Exercise 2

In [ ]:
%%sql

WITH avg_order_value AS (
    SELECT AVG(OrderDetails.UnitPrice * OrderDetails.Quantity) AS average_value
    FROM OrderDetails
)
SELECT DISTINCT customers.CompanyName
FROM customers
JOIN orders ON customers.CustomerID = orders.CustomerID
JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
WHERE (OrderDetails.UnitPrice * OrderDetails.Quantity) > (SELECT average_value FROM avg_order_value);


### Exercise 3

In [ ]:
%%sql

WITH most_ordered_products AS (
    SELECT customers.CustomerID, OrderDetails.ProductID, COUNT(*) AS order_count
    FROM customers
    JOIN orders ON customers.CustomerID = orders.CustomerID
    JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
    GROUP BY customers.CustomerID, OrderDetails.ProductID
)
SELECT customers.CompanyName, products.ProductName, max_order_count
FROM (
    SELECT CustomerID, MAX(order_count) AS max_order_count
    FROM most_ordered_products
    GROUP BY CustomerID
) AS max_order_count
JOIN most_ordered_products ON max_order_count.CustomerID = most_ordered_products.CustomerID AND max_order_count.max_order_count = most_ordered_products.order_count
JOIN customers ON most_ordered_products.CustomerID = customers.CustomerID
JOIN products ON most_ordered_products.ProductID = products.ProductID;


### Exercise 4

In [ ]:
%%sql

WITH avg_reports AS (
    SELECT AVG(report_count) AS average_count
    FROM (
        SELECT COUNT(*) AS report_count
        FROM employees
        JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
        GROUP BY employees.EmployeeID
    ) AS report_counts
)
SELECT employees.*
FROM employees
JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
GROUP BY employees.EmployeeID
HAVING COUNT(*) > (SELECT average_count FROM avg_reports);

#

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/refs/heads/master/ALX_banners/ALX_Navy.png"  style="width:100px"  ;/>
</div>